# Non-normality: when eigenvalues mislead

A matrix is **normal** when AAᵀ = AᵀA. Symmetric, skew-symmetric and orthogonal
matrices are normal. A normal matrix has orthonormal eigenvectors, and that
makes its eigenvalues perfectly conditioned: a perturbation of norm ε moves
each eigenvalue by at most ε. Everything you might predict from the
eigenvalues, such as decay rates, convergence and stability, holds.

A **non-normal** matrix is one with AAᵀ ≠ AᵀA. Its eigenvectors *can* be far
from orthogonal, though they need not be: [[1, δ], [0, 2]] is non-normal for
any δ ≠ 0, and its eigenvectors approach orthogonality as δ → 0. When they are
far from orthogonal, the eigenvalues can move by much more than ε and can badly
mispredict how the matrix behaves. For a diagonalizable matrix, A = VΛV⁻¹, the
Bauer–Fike theorem bounds the movement by κ(V)·ε, where κ(V) is the condition
number of the eigenvector matrix. A defective matrix has no such V, and its
eigenvalues can move further still: [[0, 1], [ε, 0]] has eigenvalues ±√ε.

This notebook shows the standard ways to see non-normality, with matrices from
`mtl5.generators`:

1. **The same spectrum, normal and not:** perturbed-eigenvalue clouds.
2. **Pseudospectra:** where the eigenvalues *could* be, for every ε at once.
3. **The field of values:** the set of Rayleigh quotients, and the numerical
   abscissa.
4. **Transient growth:** powers of a matrix whose eigenvalues all lie inside
   the unit disc can still grow by orders of magnitude.
5. **Eigenvalue condition numbers:** which eigenvalues are sensitive, checked
   against measured perturbations.
6. **Rounding as a perturbation:** what storing a non-normal matrix in each of
   the eight formats does to its eigenvalues.

Every random matrix has a seed, so the figures are the same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
BLUES = ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0f4a96", "#0e3d7e"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

FORMATS = ["fp16", "posit16", "takum16", "f32", "posit32", "takum32", "f64", "dd_cascade"]

print(mtl5.__version__)

## Tools

Four helpers, each taking a NumPy matrix. They are written to stand alone, so
they can later move into a plotting module.

- **`perturbed_eigenvalues`:** the eigenvalues of A + E for random E with
  ‖E‖₂ = ε, through `mtl5.eigvals`.
- **`sigma_min_grid`:** σ_min(zI − A) on a grid of complex z, the function whose
  level sets are the pseudospectra. MTL5 has no complex SVD yet, so this one
  uses NumPy's.
- **`field_of_values`:** the boundary of {xᴴAx : ‖x‖ = 1}. For each angle θ,
  the top eigenvector of the Hermitian part of e^{iθ}A gives one boundary
  point. A Hermitian n×n problem H = S + iK has the same eigenvalues as the
  real symmetric 2n×2n matrix [[S, −K], [K, S]], so `mtl5.eigh` can solve it.
- **`eigenvalue_condition`:** s(λ) = ‖x‖‖y‖ / |yᴴx| from the right eigenvectors
  x of `mtl5.eig` and the left eigenvectors y, the rows of V⁻¹.

In [ ]:
def perturbed_eigenvalues(A, eps, trials=200, seed=0):
    # Eigenvalues of A + E for `trials` random real E, each scaled to ||E||_2 = eps.
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(trials):
        E = rng.standard_normal(A.shape)
        E *= eps / np.linalg.norm(E, 2)
        out.append(mtl5.eigvals(mtl5.matrix(A + E)))
    return np.concatenate(out)


def sigma_min_grid(A, re, im):
    # sigma_min(zI - A) for every z = x + iy on the grid (NumPy's batched SVD).
    n = A.shape[0]
    Z = re[None, :] + 1j * im[:, None]
    shifted = Z[..., None, None] * np.eye(n) - A
    return np.linalg.svd(shifted, compute_uv=False)[..., -1]


def field_of_values(A, m=240):
    # Boundary points of W(A) = {x^H A x : ||x|| = 1}, for a real matrix A.
    n = A.shape[0]
    sym, skew = (A + A.T) / 2, (A - A.T) / 2
    points = []
    for theta in np.linspace(0, 2 * np.pi, m, endpoint=False):
        # Hermitian part of e^{i theta} A: S + iK with S symmetric, K antisymmetric.
        S = np.cos(theta) * sym
        K = np.sin(theta) * skew
        embed = np.block([[S, -K], [K, S]])
        _, vecs = mtl5.eigh(mtl5.matrix(embed))
        top = vecs[:, -1]
        x = top[:n] + 1j * top[n:]
        points.append(np.vdot(x, A @ x) / np.vdot(x, x))
    return np.array(points)


def eigenvalue_condition(A):
    # s(lambda_i) = ||x_i|| ||y_i|| / |y_i^H x_i|, with y_i^H the rows of V^-1.
    w, V = mtl5.eig(mtl5.matrix(A))
    Yh = np.linalg.inv(V)
    s = np.linalg.norm(V, axis=0) * np.linalg.norm(Yh, axis=1) / np.abs(np.sum(Yh.T * V, axis=0))
    return w, s

## 1. The same spectrum, normal and not

The cleanest experiment holds the eigenvalues fixed and varies only the
non-normality. Take Λ = diag(1, 2, …, 8) and a fixed random strictly upper
triangular N, and set

$$A_t = Q\,(\Lambda + t\,N)\,Q^T, \qquad Q = \texttt{randorth}(8, \text{seed}=1).$$

A_t is upper triangular in the basis Q, so its eigenvalues are exactly
1, 2, …, 8 for every t. At t = 0 it is symmetric. As t grows, the
eigenvectors lean towards each other. The **Henrici departure from normality**,
dep(A) = (‖A‖²_F − Σ|λᵢ|²)^{1/2}, measures that, and here it equals t‖N‖_F
exactly.

In [ ]:
n = 8
lam = np.arange(1.0, n + 1)
Q = g.randorth(n, seed=1).to_numpy()
N = np.triu(np.random.default_rng(2).standard_normal((n, n)), k=1)


def A_t(t):
    return Q @ (np.diag(lam) + t * N) @ Q.T


def henrici(A):
    return np.sqrt(
        max(np.linalg.norm(A, "fro") ** 2 - np.sum(np.abs(mtl5.eigvals(mtl5.matrix(A))) ** 2), 0.0)
    )


ts = (0.0, 4.0, 8.0)
eps = 1e-2
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, t in zip(axes, ts):
    A = A_t(t)
    cloud = perturbed_eigenvalues(A, eps, seed=3)
    ax.scatter(
        cloud.real,
        cloud.imag,
        s=4,
        color=SERIES[0],
        alpha=0.35,
        linewidth=0,
        zorder=2,
        label=f"eigenvalues of A + E, ‖E‖₂ = {eps:g}",
    )
    ax.scatter(
        lam,
        np.zeros(n),
        s=36,
        color=SERIES[1],
        marker="D",
        edgecolor=SURFACE,
        linewidth=1,
        zorder=3,
        label="eigenvalues of A",
    )
    ax.set_title(f"t = {t:g}", fontsize=10)
    ax.set_xlabel("Re λ")
    ax.set_xlim(-0.5, 9.5)
axes[0].set_ylabel("Im λ")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
fig.suptitle("The same eigenvalues 1..8, three degrees of non-normality", color=INK, fontsize=11)
plt.show()

print(
    f"{'t':>4s} {'||AA^T - A^TA||_F':>18s} {'dep(A)':>9s} {'t||N||_F':>9s} {'kappa(V)':>10s}"
    f" {'max displacement':>17s}"
)
for t in ts:
    A = A_t(t)
    _, V = mtl5.eig(mtl5.matrix(A))
    # Bauer-Fike bounds each PERTURBED eigenvalue's distance to the spectrum.
    cloud = perturbed_eigenvalues(A, eps, seed=3)
    shift = np.min(np.abs(cloud[:, None] - lam[None, :]), axis=1).max()
    print(
        f"{t:4g} {np.linalg.norm(A @ A.T - A.T @ A, 'fro'):18.2e} {henrici(A):9.3f}"
        f" {t * np.linalg.norm(N, 'fro'):9.3f} {np.linalg.cond(V):10.1f} {shift:17.2e}"
    )

At t = 0 the cloud is invisible: no eigenvalue moves more than 6·10⁻³, less
than ε, as it must for a normal matrix. At t = 4 the same perturbations move
eigenvalues up to 0.80 from the spectrum, 80 times ε. At t = 8 they move them
up to 2.18, more than 200 times ε, and push them off the real axis in complex
pairs.
These real perturbations can't do that to a symmetric matrix.

The table matches the theory:

- The Henrici departure equals t‖N‖_F, as the construction promises.
- κ(V) grows from 1 to about 600 and then 24,000.
- Every displacement is within the Bauer–Fike bound κ(V)·ε: 6.1 at t = 4 and
  237 at t = 8. The bound is loose, but it is the right warning.

## 2. Pseudospectra

The ε-pseudospectrum Λ_ε(A) is the set of z that are eigenvalues of *some*
A + E with ‖E‖₂ ≤ ε. Equivalently, it is the set where σ_min(zI − A) ≤ ε. So the
contour lines of σ_min(zI − A) show, for every ε at once, how far a
perturbation of that size can move the spectrum (Trefethen and Embree,
*Spectra and Pseudospectra*).

Three non-normal matrices: A₈ from section 1, `kahan(12)`, which is upper
triangular, and `frank(8)`, zoomed in on its four smallest eigenvalues. The first panel also overlays the cloud for
ε = 10⁻², which has to lie inside the 10⁻² contour.

In [ ]:
def plot_pseudospectrum(ax, A, re, im, levels=(-1, -2, -3, -4)):
    smin = sigma_min_grid(A, re, im)
    colours = [BLUES[7 - 2 * i] for i in range(len(levels))]
    cs = ax.contour(
        re, im, np.log10(smin), levels=sorted(levels), colors=colours[::-1], linewidths=1.3
    )
    ax.clabel(cs, fmt=lambda v: f"1e{int(v)}", fontsize=7)
    w = mtl5.eigvals(mtl5.matrix(A))
    ax.scatter(
        w.real, w.imag, s=22, color=SERIES[1], marker="D", edgecolor=SURFACE, linewidth=1, zorder=3
    )
    ax.set_xlim(re[0], re[-1])
    ax.set_ylim(im[0], im[-1])
    ax.set_aspect("equal")
    ax.set_xlabel("Re z")
    return smin


panels = {
    "A₈ (section 1)": (A_t(8.0), np.linspace(-1, 10, 140), np.linspace(-4, 4, 100)),
    "kahan(12)": (g.kahan(12).to_numpy(), np.linspace(-0.4, 1.6, 140), np.linspace(-0.8, 0.8, 100)),
    "frank(8), near its small eigenvalues": (
        g.frank(8).to_numpy(),
        np.linspace(-0.3, 2.2, 140),
        np.linspace(-1.0, 1.0, 100),
    ),
}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9), constrained_layout=True)
for ax, (name, (A, re, im)) in zip(axes, panels.items()):
    plot_pseudospectrum(ax, A, re, im)
    ax.set_title(name, fontsize=10)
cloud = perturbed_eigenvalues(A_t(8.0), 1e-2, seed=3)
axes[0].scatter(cloud.real, cloud.imag, s=3, color=SERIES[0], alpha=0.3, linewidth=0, zorder=2)
axes[0].set_ylabel("Im z")
fig.suptitle(
    "Contours of σ_min(zI − A) at ε = 1e-1 … 1e-4; diamonds are the eigenvalues",
    color=INK,
    fontsize=11,
)
plt.show()

# Every cloud point must lie in the 1e-2 pseudospectrum: sigma_min <= 1e-2 there.
A8 = A_t(8.0)
at_cloud = np.array([np.linalg.svd(z * np.eye(n) - A8, compute_uv=False)[-1] for z in cloud])
print(f"largest sigma_min(zI - A8) over the cloud: {at_cloud.max():.2e} (must be <= 1e-2)")

Every point of the cloud lies inside the 10⁻² contour: the largest σ_min over
the cloud is 5.5·10⁻³. For A₈, even the 10⁻⁴ contour joins the eigenvalues 3
to 7 into one region, so perturbations of that size can already make them
meet. `kahan(12)`'s twelve eigenvalues lie on the real interval [0.45, 1], yet
its 10⁻¹ pseudospectrum reaches Re z ≈ −0.25 and |Im z| ≈ 0.67.
`frank(8)`'s three smallest eigenvalues share a single 10⁻⁴ contour, and the
10⁻³ contour takes in the fourth. The eigenvalue at 1.70 has only a small
10⁻² circle of its own. Section 5 shows the same split for `frank(12)`, where
the small eigenvalues are the sensitive ones.

## 3. The field of values

The field of values W(A) = {xᴴAx : ‖x‖ = 1} is the set of Rayleigh quotients. It
is convex and contains the spectrum. For a normal matrix it is exactly the
convex hull of the eigenvalues, a segment of the real line for a symmetric one.
For a non-normal matrix it *can* bulge beyond them, as it does for A_t below.
Equality with the convex hull doesn't prove a matrix normal, though.

Its rightmost point is the **numerical abscissa** ω(A) = λ_max((A + Aᵀ)/2). That
is the initial growth rate of ‖e^{tA}‖, however negative the eigenvalues are.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, t in zip(axes, ts):
    A = A_t(t)
    W = field_of_values(A)
    closed = np.append(W, W[:1])
    ax.fill(closed.real, closed.imag, color=SERIES[0], alpha=0.12, zorder=1)
    ax.plot(
        closed.real, closed.imag, color=SERIES[0], linewidth=1.5, zorder=2, label="boundary of W(A)"
    )
    ax.scatter(
        lam,
        np.zeros(n),
        s=30,
        color=SERIES[1],
        marker="D",
        edgecolor=SURFACE,
        linewidth=1,
        zorder=3,
        label="eigenvalues",
    )
    omega = mtl5.eigvalsh(mtl5.matrix((A + A.T) / 2))[-1]
    ax.axvline(omega, color=INK_2, linewidth=1, linestyle="--", zorder=2)
    ax.text(
        omega,
        0.92,
        f" ω = {omega:.1f}",
        transform=ax.get_xaxis_transform(),
        fontsize=8,
        color=INK_2,
    )
    ax.set_title(f"t = {t:g}", fontsize=10)
    ax.set_xlabel("Re")
axes[0].set_ylabel("Im")
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle(
    "Field of values of A_t: the eigenvalues stay at 1..8, W(A) grows", color=INK, fontsize=11
)
plt.show()

for t in ts:
    A = A_t(t)
    W = field_of_values(A)
    print(
        f"t = {t:g}: W(A) spans Re {W.real.min():7.2f} .. {W.real.max():6.2f},"
        f" |Im| <= {np.abs(W.imag).max():5.2f}"
    )

At t = 0, W(A) is the segment [1, 8], the convex hull of the eigenvalues. As
t grows, it swells into a region spanning Re −3.7 to 14.9 at t = 4 and −9.9 to
25.5 at t = 8, though the eigenvalues never leave [1, 8].

Two readings follow:

- **ω(A) grows from 8 to 25.5.** For the matrix exponential, ‖e^{tA}‖ starts out
  growing at rate ω, not at the largest eigenvalue's rate.
- **W(A) reaches into the left half-plane** for t ≥ 4. Some xᵀAx are then
  negative, although every eigenvalue is positive: the symmetric part
  (A + Aᵀ)/2 is indefinite. Positive eigenvalues don't make a non-normal matrix
  positive definite as a quadratic form.

## 4. Transient growth

Eigenvalues predict the *eventual* behaviour of Aᵏ: when the spectral radius
ρ(A) is below 1, ‖Aᵏ‖ → 0. They don't predict what happens on the way. Take
the construction of section 1 with eigenvalues 0.2, 0.3, …, 0.9, so ρ = 0.9 for
every t, and compute ‖Aᵏ‖₂ with `mtl5.matmul` and `mtl5.svdvals`.

In [ ]:
lam_s = np.linspace(0.2, 0.9, n)


def B_t(t):
    return Q @ (np.diag(lam_s) + t * N) @ Q.T


ks = np.arange(0, 81)
fig, ax = plt.subplots(figsize=(7.5, 3.8), constrained_layout=True)
for slot, t in enumerate((0.0, 0.5, 1.0)):
    B = mtl5.matrix(B_t(t))
    P = mtl5.matrix(np.eye(n))
    norms = []
    for _ in ks:
        norms.append(mtl5.svdvals(P)[0])
        P = mtl5.matmul(P, B)
    ax.semilogy(ks, norms, color=SERIES[slot], linewidth=2, label=f"t = {t:g}", zorder=3)
    print(
        f"t = {t:g}: max ||A^k|| = {max(norms):9.2e} at k = {int(np.argmax(norms)):2d},"
        f" ||A^80|| = {norms[-1]:.2e}"
    )
ax.semilogy(ks, 0.9**ks, color=INK_2, linewidth=1.2, linestyle="--", label="ρᵏ = 0.9ᵏ", zorder=4)
ax.set_xlabel("k")
ax.set_ylabel("‖Aᵏ‖₂")
ax.set_title("Powers of matrices with the same eigenvalues 0.2..0.9", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.show()

For the normal matrix (t = 0), ‖Aᵏ‖₂ = 0.9ᵏ exactly, and the dashed line lies
on top of it. The non-normal ones grow first: to 6 at k = 5 for t = 0.5, and
to 92 at k = 17 for t = 1. At k = 80 the t = 1 matrix is still at 0.27, while
the normal one has decayed to 2·10⁻⁴. All three end up decaying at the same
rate, 0.9 per step, the parallel slopes at the right.

This is the practical danger of non-normality. A stationary iteration, a
time-stepping scheme or a discrete-time system with ρ < 1 converges
eventually, but the spectral radius doesn't say when, or how large the
transient will be. The pseudospectra do. If the ε-pseudospectrum contains a
point z with |z| > 1, then sup_k ‖Aᵏ‖ ≥ (|z| − 1)/ε (the Kreiss matrix theorem).
What matters is how far outside the unit disc the pseudospectrum reaches
*relative to ε*: 0.9·I has an ε-pseudospectrum reaching 0.9 + ε and no
transient at all.

## 5. Eigenvalue condition numbers

Each simple eigenvalue λ has a condition number s(λ) = ‖x‖‖y‖ / |yᴴx|, from its
right eigenvector x and left eigenvector y. To first order, a perturbation E
moves λ by at most s(λ)‖E‖₂. For a normal matrix every s(λ) is 1. The chart
shows s(λ) for four matrices, each eigenvalue a bar.

In [ ]:
cond_mats = {
    "A₀ (normal)": A_t(0.0),
    "A₈": A_t(8.0),
    "kahan(12)": g.kahan(12).to_numpy(),
    "frank(12)": g.frank(12).to_numpy(),
}
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), sharey=True, constrained_layout=True)
for ax, (name, A) in zip(axes, cond_mats.items()):
    w, s = eigenvalue_condition(A)
    order = np.argsort(w.real)
    ax.bar(np.arange(len(w)), s[order], color=SERIES[0], width=0.7, zorder=3)
    ax.set_yscale("log")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("eigenvalue, by real part")
    ax.set_xticks([])
    print(f"{name:12s} s(lambda) from {s.min():8.1f} to {s.max():9.2e}")
axes[0].set_ylabel("s(λ)")
plt.show()

Every bar of the normal A₀ is exactly 1. A₈ ranges from 94 to 6,500, and
`kahan(12)` from 17 to 32,000. `frank(12)` spans more than seven decades, from
1.7 for its largest eigenvalues to 3.9·10⁷ for its smallest: Frank's small
eigenvalues are the sensitive ones.

The condition numbers make a testable prediction. Perturb `frank(12)` by a
random E with ‖E‖₂ = 10⁻¹⁰ and measure how far each eigenvalue moves: every
displacement should be at most about s(λ)·10⁻¹⁰.

In [ ]:
F = g.frank(12).to_numpy()
w0, s0 = eigenvalue_condition(F)
E = np.random.default_rng(5).standard_normal(F.shape)
eps_f = 1e-10
E *= eps_f / np.linalg.norm(E, 2)
w1 = mtl5.eigvals(mtl5.matrix(F + E))
moved = np.array([np.min(np.abs(w1 - lam_i)) for lam_i in w0])

fig, ax = plt.subplots(figsize=(5.6, 4.0), constrained_layout=True)
lim = np.array([1e-12, 1e-2])
ax.loglog(lim, lim, color=RULE, linewidth=1.5, linestyle="--", label="displacement = s(λ)·ε")
ax.loglog(
    s0 * eps_f,
    moved,
    linestyle="none",
    marker="o",
    markersize=6,
    color=SERIES[0],
    label="eigenvalues of frank(12)",
)
ax.set_xlabel("predicted bound s(λ)·ε")
ax.set_ylabel("measured |Δλ|")
ax.set_title("Perturbing frank(12) by ‖E‖₂ = 1e-10", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

# Frank's eigenvalues come in exact reciprocal pairs, lambda and mu = 1/lambda.
# To first order, lambda*mu - 1 = mu*dlambda + lambda*dmu = dlambda/lambda +
# dmu/mu, and each |d| is at most s * u * ||F||_2 for float64 rounding.
order = np.argsort(w0.real)
w_sorted, s_sorted = w0.real[order], s0[order]
scale = 2.0**-53 * np.linalg.norm(F, 2)
for i in range(3):
    lam_i, mu_i = w_sorted[i], w_sorted[-1 - i]
    bound = s_sorted[i] * scale / lam_i + s_sorted[-1 - i] * scale / mu_i
    print(
        f"lambda_{i + 1} * lambda_{12 - i} - 1 = {lam_i * mu_i - 1:+.1e}"
        f"   first-order bound {bound:.1e}"
    )

Every point lies on or below the line: the first-order bound holds, with the
measured displacements typically a decade under it. The eigenvalues with
s(λ) ≈ 10⁷ move by 10⁻⁴ under a perturbation of 10⁻¹⁰.

The reciprocal pairs give a check that needs no perturbation at all. In exact
arithmetic, each product λ·μ of a pair is exactly 1. An error δλ in λ and δμ in
μ change the product by about μ·δλ + λ·δμ = δλ/λ + δμ/μ, and each δ is at most
s·u‖F‖ for float64 rounding. The computed products miss by up to 1.9·10⁻⁷,
inside that first-order bound, which is a few times 10⁻⁶ for the smallest
pairs. The misses are as large as they are because the small eigenvalues are
ill conditioned (s ≈ 10⁷), not because the eigensolver is careless.

## 6. Rounding as a perturbation

Storing a matrix in a format rounds every entry, a perturbation with ‖E‖₂ of
order u‖A‖. So the eigenvalues of the *stored* matrix can sit up to about
s(λ)·u‖A‖ from the true ones, before any arithmetic happens. The experiment
rounds A₈ to each of the eight formats and computes the eigenvalues of the
stored matrix in float64. It compares them with the exact eigenvalues 1, …, 8,
which the construction fixes independently of any solver.

In [ ]:
A8 = A_t(8.0)
_, s8 = eigenvalue_condition(A8)
normA = np.linalg.norm(A8, 2)


def unit_roundoff(dtype):
    for k in range(1, 53):
        if mtl5.convert(np.array([1.0 + 2.0**-k]), dtype).to_numpy()[0] == 1.0:
            return 2.0**-k
    return 2.0**-53


print(f"{'format':11s} {'u':>8s} {'max |dlambda|':>14s} {'s_max u ||A||':>14s}")
for name in FORMATS:
    stored = mtl5.convert(A8, name).to_numpy()
    # A tight tolerance: at the default 1e-10 the eigensolver itself stops
    # around 1e-9 here, which would swamp the float64 rows.
    w_stored = mtl5.eigvals(mtl5.matrix(stored), 1e-16)
    # Distance of each computed eigenvalue of the stored matrix to the exact
    # spectrum 1..8, which the construction fixes independently of any solver.
    moved = np.min(np.abs(w_stored[:, None] - lam[None, :]), axis=1).max()
    if np.array_equal(stored, A8):
        # float64 data stores exactly in f64 and dd_cascade: storage adds nothing.
        print(f"{name:11s} {'':>8s} {moved:14.2e}   storage exact (no rounding)")
        continue
    u = unit_roundoff(name)
    print(f"{name:11s} {u:8.1e} {moved:14.2e} {s8.max() * u * normA:14.2e}")

Every displacement is within the s_max·u‖A‖ bound, and the formats separate
cleanly:

- **16 bits:** the stored eigenvalues move by 1.1 to 1.4, more than the
  spacing between them. The matrix a 16-bit format holds doesn't have the
  spectrum 1, 2, …, 8 in any useful sense.
- **32 bits:** f32 moves them by 4·10⁻³. posit32 and takum32 move them by
  3·10⁻⁵ and 8·10⁻⁵, two orders of magnitude better: their extra precision
  near 1 compounded by the conditioning.
- **f64 and `dd_cascade`:** storage is exact, so the remaining 6.5·10⁻¹² is
  the combined error of forming A₈ in float64 and of the eigensolver. It is
  within s_max·u‖A‖ = 2.5·10⁻¹¹, the amount that rounding Q·(Λ + 8N)·Qᵀ in
  float64 could cause by itself, and LAPACK's `numpy.linalg.eigvals` gives the
  same 7·10⁻¹². At `eigvals`' default tolerance of 10⁻¹⁰ the eigensolver would
  stop at 1.4·10⁻⁹ instead, hence the tight tolerance above.

s(λ)·u is the eigenvalue counterpart of the κ·u rule in the conditioning
notebook. It says how many digits of an eigenvalue a format can hold before
any arithmetic is done.

## What to take away

- **Normality guarantees well-conditioned eigenvalues; non-normality calls for
  a check.** Some non-normal matrices have well-conditioned eigenvalues too,
  like [[1, δ], [0, 2]] for small δ. For a non-normal matrix, look at κ(V), the
  eigenvalue condition numbers or the pseudospectra before trusting what the
  eigenvalues predict.
- **The pictures answer different questions.**
  - Perturbed clouds and pseudospectra: where can the eigenvalues go?
  - The field of values: what is the initial growth rate, and what are the
    Rayleigh quotients?
  - Transient growth: how large does Aᵏ get before it decays?
  - s(λ): which eigenvalues are sensitive?
- **Non-normality is a controlled variable.** `randorth(n, seed=)` and a
  triangular factor give any spectrum with any degree of non-normality. Catalog
  matrices such as `kahan` and `frank` give the classic cases.

### Where to go next

- `conditioning.ipynb` and `spectra_inertia.ipynb`, the earlier notebooks of
  #166.
- Coming in #166: the stability of factorizations, and Krylov convergence
  against the shape of the spectrum, where non-normality returns through
  GMRES.